In [24]:
import torch 
def layernorm(x, gamma, beta, eps = 1e-5):
    # (x-mean)/ sqrt(var+ eps) * gamma  + beta 
    mean = torch.mean(x, dim= -1, keepdim=True)
    var = torch.var(x, dim = -1, keepdim=True, unbiased = False )
    x_norm = (x-mean) / torch.sqrt(var+eps)
    return x_norm * gamma + beta 
x = torch.rand(4, 5, 2)
gamma=torch.ones(2)
beta=torch.zeros(2)
x_norm1 = layernorm(x,gamma, beta) 
print(x_norm1.shape)

from torch.nn  import LayerNorm
x_norm = LayerNorm(2 )(x)
torch.allclose(x_norm1, x_norm)

torch.Size([4, 5, 2])


True

# Layer Norm — Revision Notes

## Why normalize at all?
- Activations can blow up across layers → gradients explode during backprop
- Keeps activations in a stable range for both training and inference

## Why not Batch Norm for transformers?
- Normalizes across batch axis → needs large, consistent batches
- Padding corrupts statistics (averaging real tokens with garbage pad tokens)
- Batch size 1 at inference → meaningless statistics
- Requires running mean/var → different behavior train vs inference

## Why Layer Norm?
- Normalizes across d_model (feature dimension) per token
- Each token normalizes itself independently — no dependency on other tokens, other samples, or batch size
- Same computation at training and inference (no running statistics, no mode switching)

## The formula
```
mean = mean(x, dim=-1)          # shape: (batch, seq_len, 1)
var  = var(x, dim=-1)           # shape: (batch, seq_len, 1)
out  = (x - mean) / sqrt(var + eps) * gamma + beta
```

## Gamma and beta
- Learnable parameters, shape `(d_model,)`
- One per feature, shared across all tokens and samples
- Purpose: after normalizing to mean=0/var=1, let the network learn back the optimal distribution per feature
- Initialized to gamma=1, beta=0 (identity at start)
- Learned during training, frozen at inference (like any weight)
- Each LayerNorm module has its own independent gamma/beta

## Why `unbiased=False`?
- You have all d_model features — that's the full population, not a sample
- No Bessel's correction needed → divide by N, not N-1

## Pre-LN vs Post-LN
- **Post-LN** (original): Attention → Add residual → LayerNorm
- **Pre-LN** (modern/GPT-2+): LayerNorm → Attention → Add residual
- Pre-LN keeps the residual stream clean — raw x flows through, attention output is just a delta
- Gradients flow through the addition unmodified (gradient of add = 1) → stable training in deep models
- Post-LN forces gradients through LayerNorm on the residual path → can squash/amplify them at depth

False